# Submit the Single-Step Merge Job

Load the workshop-local command-job YAML, replace its compute placeholder from `.env`, submit it, stream logs, and verify completion.

**Source:** Adapted from this repository's `notebooks/00_submit_azureml_pipelines.ipynb` and `pipelines/single-step-merge-job.yaml`.

In [1]:
from pathlib import Path
import os

from azure.ai.ml import MLClient, load_job
from azure.ai.ml.entities import ManagedIdentityConfiguration
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
COMPUTE_NAME = os.environ["AZUREML_COMPUTE_NAME"]
COMPUTE_IDENTITY_CLIENT_ID = os.environ["AZUREML_COMPUTE_IDENTITY_CLIENT_ID"].strip()
if not COMPUTE_IDENTITY_CLIENT_ID:
    raise ValueError("AZUREML_COMPUTE_IDENTITY_CLIENT_ID must identify the compute cluster UMI")
RUN_JOB = os.getenv("RUN_SINGLE_STEP_JOB", "false").lower() in {"1", "true", "yes"}
pipeline_path = WORKSHOP_ROOT / "pipelines/single-step-merge-job.yaml"

In [ ]:
job = load_job(pipeline_path)
job.compute = COMPUTE_NAME
job.identity = ManagedIdentityConfiguration(client_id=COMPUTE_IDENTITY_CLIENT_ID)
job.display_name = "Workshop single-step taxi merge"
job.tags = {"workshop": "azureml-h2o", "operation": "single-step-merge"}

assert isinstance(job.identity, ManagedIdentityConfiguration)
assert job.identity.client_id == COMPUTE_IDENTITY_CLIENT_ID
print(f"Loaded: {type(job).__name__}")
print(f"Compute: {COMPUTE_NAME}")
print(f"Runtime identity: compute cluster UMI {COMPUTE_IDENTITY_CLIENT_ID}")

if RUN_JOB:
    submitted_job = ml_client.jobs.create_or_update(job)
    print(f"Submitted: {submitted_job.name}")
    ml_client.jobs.stream(submitted_job.name)
    final_job = ml_client.jobs.get(submitted_job.name)
    if final_job.status != "Completed":
        raise RuntimeError(f"Job ended with status {final_job.status}")

    output_datastore = os.getenv("AZUREML_OUTPUT_DATASTORE", "workspaceblobstore")
    output_uri = (
        f"azureml://datastores/{output_datastore}/paths/azureml/"
        f"{final_job.name}/merged_data/"
    )
    download_dir = WORKSHOP_ROOT / "outputs/single_step_merge" / final_job.name
    ml_client.jobs.download(
        final_job.name,
        output_name="merged_data",
        download_path=download_dir,
    )
    merged_path = download_dir / "named-outputs/merged_data/merged_taxi_data.csv"
    if not merged_path.is_file():
        raise FileNotFoundError(f"Expected downloaded output at {merged_path}")

    print(f"Studio: {final_job.studio_url}")
    print(f"Merged output URI: {output_uri}")
    print(f"Downloaded merged data: {merged_path}")
else:
    print("Submission disabled. Set RUN_SINGLE_STEP_JOB=true in workshop/.env.")

## Expected Result

The command job completes on the configured cluster and publishes a merged taxi CSV as its `merged_data` output.

Next: `02_submit_integration_compare.ipynb`.